# Decision Trees - Hands-On Exercises 🌳

Welcome to the Decision Trees exercises! This notebook builds practical intuition for impurity measures, recursive splitting, overfitting, regression trees, tuning, and feature importance.

## 🎯 Learning Objectives
- Implement and compare Gini impurity and entropy
- Train and inspect the from-scratch `DecisionTreeScratch` implementation
- Compare the custom implementation with scikit-learn
- Diagnose overfitting using tree depth and validation scores
- Compare Gini and entropy criteria
- Apply decision trees to regression
- Tune hyperparameters with cross-validation
- Interpret feature importance

## 📚 Prerequisites
- Basic classification and regression concepts
- Python, NumPy, pandas, and matplotlib
- Familiarity with impurity and information gain

## 🗂️ Exercise Structure
1. Impurity measures
2. First from-scratch tree
3. Custom vs. scikit-learn
4. Overfitting and `max_depth`
5. Gini vs. entropy
6. Regression trees
7. Hyperparameter tuning
8. Feature importance

Each exercise has a task, a TODO cell, and hints. Try the TODO before reading the hints.

## 🔧 Setup and Imports

The setup below loads `implementation.py` from this folder so that every exercise uses the local from-scratch implementation.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.datasets import make_classification, make_regression, load_iris

# Load the from-scratch implementation from this folder
impl_path = Path.cwd() / 'implementation.py'
if not impl_path.exists():
    impl_path = Path('01_supervised_learning/03_decision_trees/implementation.py').resolve()

spec = importlib.util.spec_from_file_location('decision_tree_impl', impl_path)
dt_impl = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dt_impl)

DecisionTreeScratch = dt_impl.DecisionTreeScratch
generate_classification_data = dt_impl.generate_classification_data
generate_regression_data = dt_impl.generate_regression_data

np.random.seed(42)
plt.style.use('default')
sns.set_palette('husl')

print("✅ All libraries imported successfully!")
print(f"📦 Loaded implementation from: {impl_path}")
print("🌳 Ready to start Decision Tree exercises!")

---

# Exercise 1: Impurity Measures 🌱

A decision tree chooses the split that reduces impurity the most.

**Task**: Implement Gini impurity and entropy, then test them on pure, balanced, imbalanced, and multi-class label sets.

**Formulas**
- Gini: \(1 - \sum_i p_i^2\)
- Entropy: \(-\sum_i p_i \log_2(p_i)\)

**Hint**: Use `np.unique(y, return_counts=True)` to count classes. Convert counts to proportions and drop zero proportions before taking logarithms.

In [ ]:
def gini_impurity(y):
    '''Return Gini impurity for an array of class labels.

    TODO:
    - Count each class with np.unique(y, return_counts=True)
    - Convert counts to proportions
    - Return 1 - sum(p_i ** 2)
    '''
    if len(y) == 0:
        return 0.0
    # YOUR CODE HERE
    pass


def entropy(y):
    '''Return Shannon entropy (base 2) for an array of class labels.

    TODO:
    - Count each class with np.unique(y, return_counts=True)
    - Convert counts to proportions
    - Return -sum(p_i * log2(p_i))
    '''
    if len(y) == 0:
        return 0.0
    # YOUR CODE HERE
    pass


test_cases = [
    [0, 0, 0, 0],          # Pure
    [0, 1, 0, 1],          # Balanced binary
    [0, 0, 0, 1],          # Imbalanced binary
    [0, 1, 2, 0, 1, 2],    # Multi-class
]

print("Impurity Measure Tests:")
print("Labels\t\t\tGini\tEntropy")
print("-" * 45)
for labels in test_cases:
    gini = gini_impurity(labels)
    ent = entropy(labels)
    print(f"{str(labels):24s}{gini:.4f}\t{ent:.4f}")

**Check your work**: compare with the private helpers `_gini_impurity` and `_entropy` on `DecisionTreeScratch`.

**Expected output analysis**
- A pure node has Gini = 0 and entropy = 0.
- A balanced binary node has Gini = 0.5 and entropy = 1.0.
- An imbalanced node has low impurity but not zero.

In [ ]:
# TODO: instantiate a DecisionTreeScratch model with task_type='classification'
model = None

for labels in test_cases:
    # TODO: compare your functions with model._gini_impurity and model._entropy
    # YOUR CODE HERE
    pass

---

# Exercise 2: Train Your First From-Scratch Tree 🛠️

**Task**
1. Generate data with `generate_classification_data(n_samples=600, random_state=42)`.
2. Split it with stratification.
3. Train a `DecisionTreeScratch` classifier.
4. Report train/test accuracy, depth, and number of leaves.
5. Print the learned tree.

**Hint**: Use `DecisionTreeScratch(task_type='classification', criterion='gini', max_depth=4, min_samples_split=10, min_samples_leaf=5, random_state=42)`. Methods: `.fit()`, `.score()`, `.get_depth()`, `.get_n_leaves()`, `.print_tree()`.

In [ ]:
X_clf, y_clf = generate_classification_data(n_samples=600, random_state=42)

# TODO: split into training and test sets (use stratify=y_clf)
# YOUR CODE HERE
X_train = X_test = y_train = y_test = None

tree = DecisionTreeScratch(
    task_type='classification',
    criterion='gini',
    max_depth=4,            # Try 1, 3, 8, or None later
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
)

# TODO: fit the tree and compute train/test accuracy
# YOUR CODE HERE

# TODO: print accuracy, depth, number of leaves, and the tree structure
# YOUR CODE HERE

**Expected output analysis**
- Training accuracy is usually higher than test accuracy.
- Increasing `max_depth` increases leaves and overfitting risk.
- A tree with `max_depth=1` underfits and has high bias.

**Hint**: Use `tree.fit(X_train, y_train)` and `tree.score(X, y)`.

---

# Exercise 3: Compare the Custom Tree with Scikit-Learn 📊

**Task**: Complete `compare_implementations` to train both models, evaluate them, and return a dictionary of results.

**Hint**
- Use matching `max_depth`, `min_samples_split`, `min_samples_leaf`, and `random_state`.
- Compare test accuracy, depth, and number of leaves.
- Small differences are normal because the split-search implementations differ.

In [ ]:
def compare_implementations(X, y, test_size=0.2, random_state=42):
    '''Compare DecisionTreeScratch with sklearn's DecisionTreeClassifier.

    TODO:
    - Split X and y into train/test sets
    - Train a custom DecisionTreeScratch classifier
    - Train a sklearn DecisionTreeClassifier with matching hyperparameters
    - Return accuracy, depth, and number of leaves for both models
    '''
    # TODO: split the data
    # YOUR CODE HERE

    # TODO: train and evaluate the custom implementation
    # YOUR CODE HERE

    # TODO: train and evaluate scikit-learn
    # YOUR CODE HERE

    results = {
        'custom': {'train_accuracy': None, 'test_accuracy': None,
                   'depth': None, 'leaves': None},
        'sklearn': {'train_accuracy': None, 'test_accuracy': None,
                    'depth': None, 'leaves': None},
    }
    return results


results = compare_implementations(X_clf, y_clf)

print("Model Comparison")
print("-" * 60)
print(f"{'Model':<12}{'Train Acc':<12}{'Test Acc':<12}{'Depth':<8}{'Leaves':<8}")
print("-" * 60)
for name, metrics in results.items():
    print(f"{name:<12}{metrics['train_accuracy']:<12.4f}"
          f"{metrics['test_accuracy']:<12.4f}{metrics['depth']:<8}{metrics['leaves']:<8}")

**Discussion**
- Are the accuracies close? A difference of a few percentage points is expected.
- How does the number of leaves relate to complexity?
- Which model is easier to inspect?

**Hint**: The custom tree exposes `.get_depth()` and `.get_n_leaves()`; sklearn exposes `.get_depth()` and `.get_n_leaves()` as well.

---

# Exercise 4: Overfitting and `max_depth` 📈

**Task**: Train the custom tree for depths 1 through 12 and plot training and test accuracy.

**Hint**
- Keep `min_samples_split` and `min_samples_leaf` fixed so only depth changes.
- Store the scores in lists inside the loop.
- Use `plt.plot`, labels, legend, and grid.

In [ ]:
depths = range(1, 13)
train_scores = []
test_scores = []

for depth in depths:
    # TODO: create a DecisionTreeScratch with max_depth=depth
    # TODO: fit it on the training data
    # TODO: record training and test accuracy
    # YOUR CODE HERE
    pass

# TODO: plot training and test accuracy against tree depth
# YOUR CODE HERE

# TODO: print the depth with the best test accuracy
# YOUR CODE HERE

**Expected output analysis**
- Training accuracy should increase with depth and eventually reach 1.0.
- Test accuracy may rise, flatten, or fall as the tree memorizes noise.
- A large train-test gap signals overfitting.

**Hint**: Use `np.argmax(test_scores)` and index into `list(depths)`.

---

# Exercise 5: Gini vs. Entropy ⚖️

**Task**: Train trees with `criterion='gini'` and `criterion='entropy'` on the same data and compare accuracy and complexity.

**Hint**
- Keep all other hyperparameters fixed.
- Entropy can produce slightly different splits and may be a little slower.
- Use both `DecisionTreeScratch` and `DecisionTreeClassifier` if you want a broader comparison.

In [ ]:
criteria = ['gini', 'entropy']
criterion_results = {}

for criterion in criteria:
    # TODO: train a custom tree with this criterion
    # TODO: record test accuracy, depth, and number of leaves
    # YOUR CODE HERE
    pass

print("Splitting Criterion Comparison")
print("-" * 50)
for criterion, metrics in criterion_results.items():
    # TODO: print the results
    # YOUR CODE HERE
    pass

**Discussion**
- Did one criterion produce a smaller tree?
- Were test accuracies meaningfully different?
- Is the result stable if you change the random seed?

**Hint**: There is no universally better criterion; the best choice depends on the data.

---

# Exercise 6: Regression Trees 🏠

**Task**
1. Generate data with `generate_regression_data(n_samples=500, noise=0.5, random_state=42)`.
2. Train a custom regression tree and a scikit-learn `DecisionTreeRegressor`.
3. Compute MSE and \(R^2\) for both models.
4. Plot actual vs. predicted values for the custom tree.

**Hint**
- Use `DecisionTreeScratch(task_type='regression', criterion='mse', ...)`.
- The custom `.score()` returns \(R^2\) for regression.
- Use `mean_squared_error` and `r2_score` from scikit-learn.

In [ ]:
X_reg, y_reg = generate_regression_data(n_samples=500, noise=0.5, random_state=42)

# TODO: split into training and test sets
# YOUR CODE HERE

# TODO: train the custom regression tree
# YOUR CODE HERE

# TODO: train the sklearn regression tree
# YOUR CODE HERE

# TODO: compute predictions, MSE, and R² for both models
# YOUR CODE HERE

# TODO: plot actual vs. predicted values for the custom tree
# YOUR CODE HERE

**Expected output analysis**
- Deeper trees fit training data better but may not generalize.
- Regression trees predict piecewise-constant values.
- A reasonable model explains a substantial fraction of target variance.

**Hint**: Use `plt.scatter(y_test, y_pred, alpha=0.6)` and add the 45-degree line `plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')`.

---

# Exercise 7: Hyperparameter Tuning with Grid Search 🎛️

**Task**
1. Create a synthetic classification dataset.
2. Define a grid over `max_depth`, `min_samples_leaf`, and `criterion`.
3. Run `GridSearchCV` with 5-fold cross-validation.
4. Print the best parameters and best CV score.
5. Train a custom tree with the winning parameters and compare test accuracy.

**Hint**
- Use `GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)`.
- Access the best model with `grid_search.best_estimator_`.
- Keep the grid small enough to run quickly.

In [ ]:
X_tune, y_tune = make_classification(
    n_samples=600,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=3,
    random_state=42,
)

# TODO: split into training and test sets
# YOUR CODE HERE

# TODO: define a parameter grid
param_grid = {
    # 'max_depth': [...],
    # 'min_samples_leaf': [...],
    # 'criterion': [...],
}

# TODO: run GridSearchCV
# YOUR CODE HERE

# TODO: print the best parameters and best CV score
# YOUR CODE HERE

# TODO: train a custom tree with the best parameters and compare test accuracy
# YOUR CODE HERE

**Discussion**
- Which hyperparameter had the largest effect?
- Does the best CV score match test accuracy?
- Why should tuning not use the test set?

**Hint**: Use `grid_search.best_params_` and `grid_search.best_score_`.

---

# Exercise 8: Feature Importance and Interpretation 🔍

**Task**
1. Load the Iris dataset.
2. Train a custom tree and a scikit-learn tree.
3. Extract and sort feature importances.
4. Plot the top features for both models.
5. Identify the most important feature.

**Hint**
- `model.feature_importances_` aligns with the feature columns.
- Use `np.argsort(importances)[::-1]` for descending order.
- Use `iris.feature_names` for readable labels.

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
feature_names = iris.feature_names

# TODO: split the data
# YOUR CODE HERE

# TODO: train the custom tree
# YOUR CODE HERE

# TODO: train the sklearn tree
# YOUR CODE HERE

# TODO: create sorted importance DataFrames for both models
# YOUR CODE HERE

# TODO: plot the importances (bar chart)
# YOUR CODE HERE

# TODO: print the most important feature for each model
# YOUR CODE HERE

**Expected output analysis**
- Petal width and petal length are usually the most important Iris features.
- The custom and sklearn rankings should be broadly similar.
- Feature importance measures impurity reduction, not causality.

**Hint**: A horizontal bar chart (`barh`) is often easier to read when feature names are long.

---

# 📚 Summary and Next Steps

## What You've Practiced
✅ Implementing and comparing impurity measures  
✅ Training and inspecting a from-scratch tree  
✅ Comparing the custom implementation with scikit-learn  
✅ Diagnosing overfitting with `max_depth`  
✅ Comparing Gini and entropy  
✅ Training regression trees  
✅ Tuning hyperparameters with cross-validation  
✅ Interpreting feature importance  

## Next Steps
1. **Random Forests**: Combine many trees to reduce variance.
2. **Gradient Boosting**: Build trees sequentially to reduce bias.
3. **Pruning**: Explore cost-complexity pruning with `ccp_alpha`.
4. **Real-world data**: Handle categorical features, missing values, and imbalanced classes.
5. **Model interpretation**: Learn about partial dependence and SHAP values.

## Resources
- Scikit-learn Decision Trees: https://scikit-learn.org/stable/modules/tree.html
- *The Elements of Statistical Learning* by Hastie, Tibshirani, and Friedman
- *Hands-On Machine Learning* by Aurélien Géron

---

**🎊 Congratulations on completing the Decision Trees exercises!**  
Experiment with different datasets and hyperparameters to build intuition for how trees learn.